# 🏷️ AraT5 Hierarchical Text Classifier

**Model:** [UBC-NLP/AraT5-base](https://huggingface.co/UBC-NLP/AraT5-base)  
**Task:** Hierarchical classification (Main Category → Subcategory)  
**Dataset:** Almersal Arabic Articles  
**Runtime:** Google Colab Free (Tesla T4 · 16GB VRAM)  

---

### 📋 Notebook Sections

| # | Section | Description |
|---|---------|-------------|
| 1 | Setup & Config | Install libraries, mount Drive, configure paths |
| 2 | Data Preparation | Load raw data, convert to T5 format, create clean JSONL |
| 3 | Tokenization | Load tokenizer, add special tokens, tokenize dataset |
| 4 | Model Setup | Load model, resize embeddings, enable optimizations |
| 5 | Training | Configure Trainer, train with checkpoint resume support |
| 6 | Evaluation | Full evaluation with classification report & confusion matrix |
| 7 | Inference | Predict function for new articles |
| 8 | Export | Save to Drive + optional HuggingFace Hub upload |

---
## 1. ⚙️ Setup & Configuration

In [ ]:
# ============================================================
# 1.1 Install Required Libraries
# ============================================================

!pip -q install -U transformers datasets sentencepiece accelerate evaluate sacrebleu scikit-learn peft

In [ ]:
# ============================================================
# 1.2 Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 1.3 Configuration (Edit these paths as needed)
# ============================================================

import os

# ----- Dataset Paths -----
DATASET_DIR = "/content/drive/MyDrive/almersal_dataset"

# Raw data files (original JSON with all columns)
RAW_TRAIN = os.path.join(DATASET_DIR, "train.json")
RAW_VALID = os.path.join(DATASET_DIR, "validation.json")
RAW_TEST  = os.path.join(DATASET_DIR, "test.json")

# Converted T5 format files (will be created if not found)
T5_TRAIN = os.path.join(DATASET_DIR, "train_t5_clean.jsonl")
T5_VALID = os.path.join(DATASET_DIR, "validation_t5_clean.jsonl")
T5_TEST  = os.path.join(DATASET_DIR, "test_t5_clean.jsonl")

# Tokenized dataset cache
TOKENIZED_DIR = os.path.join(DATASET_DIR, "tokenized_dataset")

# ----- Model Paths -----
MODEL_NAME = "UBC-NLP/AraT5-base"
OUTPUT_DIR = "/content/drive/MyDrive/arat5_classifier"
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, "final_model")

# ============================================================
# 🎯 TRAINING STRATEGY (Choose ONE)
# ============================================================
#
#   "lora"           → 🥇 (Recommended) Train ~2% of params
#                       Fastest (3-5h), lowest memory (~6GB)
#                       Best for Colab T4 free tier
#
#   "freeze_encoder" → 🥈 Freeze encoder, train decoder
#                       Medium speed (5-7h), ~10GB memory
#                       Good balance of speed and accuracy
#
#   "full"           → 🥉 Full fine-tuning (all 220M params)
#                       Slowest (24-36h), needs resume
#                       Best accuracy but won't fit in 1 session
#
TRAINING_STRATEGY = "lora"  # <<< CHANGE THIS

# ----- LoRA Hyperparameters (only used if strategy = "lora") -----
LORA_R = 16              # LoRA rank (8-64, higher = more params)
LORA_ALPHA = 32          # LoRA alpha (usually 2x rank)
LORA_DROPOUT = 0.05      # Dropout for regularization

# ----- Training Hyperparameters -----
NUM_EPOCHS = 4
LEARNING_RATE = 3e-5 if TRAINING_STRATEGY != "lora" else 1e-4  # LoRA uses higher LR
TRAIN_BATCH_SIZE = 4 if TRAINING_STRATEGY != "lora" else 8     # LoRA allows bigger batch
EVAL_BATCH_SIZE = 8 if TRAINING_STRATEGY != "lora" else 16
GRADIENT_ACCUMULATION_STEPS = 4 if TRAINING_STRATEGY != "lora" else 2
WARMUP_STEPS = 500 if TRAINING_STRATEGY != "lora" else 200
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 16
SEED = 42

# ----- Sampling (set to 1.0 to use full dataset) -----
SAMPLE_RATIO = 1.0  # e.g., 0.1 = use 10% of data, 1.0 = full

# ----- Checkpoint Resume -----
RESUME_FROM_CHECKPOINT = True

# ----- Computed Values -----
effective_batch = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

strategy_info = {
    "lora": (
        "LoRA (PEFT)",
        "~2-5% params | ~3-5 hours | ~6GB VRAM"
    ),
    "freeze_encoder": (
        "Freeze Encoder",
        "~50% params | ~5-7 hours | ~10GB VRAM"
    ),
    "full": (
        "Full Fine-tune",
        "100% params | ~24-36 hours | ~14GB VRAM"
    )
}

name, info = strategy_info[TRAINING_STRATEGY]

print("=" * 55)
print(f"  Configuration Loaded")
print("=" * 55)
print(f"  Strategy       : {name}")
print(f"  Estimate       : {info}")
print(f"  Dataset dir    : {DATASET_DIR}")
print(f"  Output dir     : {OUTPUT_DIR}")
print(f"  Model          : {MODEL_NAME}")
print(f"  Sample ratio   : {SAMPLE_RATIO}")
print(f"  Effective batch: {effective_batch}")
print(f"  Learning rate  : {LEARNING_RATE}")
print("=" * 55)

In [ ]:
# ============================================================
# 1.4 Environment Check
# ============================================================

import torch
import random
import numpy as np

# Set reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("=" * 55)
print(f"  PyTorch Version  : {torch.__version__}")
print(f"  CUDA Available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  GPU              : {gpu_name}")
    print(f"  VRAM             : {vram_gb:.1f} GB")

    # Detect precision support
    capability = torch.cuda.get_device_capability(0)
    supports_bf16 = capability[0] >= 8  # Ampere+
    supports_fp16 = capability[0] >= 7  # Volta+ (T4 = 7.5)
    print(f"  Compute Cap.     : {capability[0]}.{capability[1]}")
    print(f"  bf16 Support     : {'✅ Yes' if supports_bf16 else '❌ No (needs Ampere+)'}")
    print(f"  fp16 Support     : {'✅ Yes' if supports_fp16 else '❌ No'}")
else:
    gpu_name = None
    supports_bf16 = False
    supports_fp16 = False
    print("  ⚠️  Running on CPU - Training will be very slow!")

print("=" * 55)
print(f"  Seed             : {SEED} ✅")

---
## 2. 📊 Data Preparation

This section handles two scenarios:
1. **Clean JSONL files exist** → Load them directly
2. **Only raw JSON files exist** → Convert to T5 format first

In [ ]:
# ============================================================
# 2.1 Data Conversion Functions
# ============================================================

import json


def build_input(article):
    """Build the input prompt for T5."""

    # Auto-detect column names
    title_key = next(
        (k for k in ['title', 'Title', 'عنوان'] if k in article),
        None
    )
    content_key = next(
        (k for k in ['content', 'Content', 'text', 'محتوى'] if k in article),
        None
    )

    if not title_key or not content_key:
        raise KeyError(
            f"Cannot find title/content columns. "
            f"Available: {list(article.keys())}"
        )

    return (
        "المهمة: تصنيف هرمي\n\n"
        f"العنوان:\n{article[title_key]}\n\n"
        f"النص:\n{article[content_key]}"
    )


def build_target(article):
    """Build the target label for T5."""

    cat_key = next(
        (k for k in ['category', 'Category', 'تصنيف'] if k in article),
        None
    )
    sub_key = next(
        (k for k in ['subcategory', 'Subcategory', 'تصنيف_فرعي'] if k in article),
        None
    )

    if not cat_key or not sub_key:
        raise KeyError(
            f"Cannot find category/subcategory columns. "
            f"Available: {list(article.keys())}"
        )

    return (
        f"<main_class>{article[cat_key]} "
        f"<sub_class>{article[sub_key]}"
    )


def convert_raw_to_t5_jsonl(input_file, output_file):
    """Convert raw JSON dataset to clean T5 JSONL format."""

    print(f"  Converting: {os.path.basename(input_file)}")

    with open(input_file, "r", encoding="utf-8") as f:
        articles = json.load(f)

    count = 0
    with open(output_file, "w", encoding="utf-8") as out:
        for article in articles:
            clean_item = {
                "input": build_input(article),
                "target": build_target(article)
            }
            out.write(
                json.dumps(clean_item, ensure_ascii=False) + "\n"
            )
            count += 1

    print(f"  ✅ Saved {count:,} samples → {os.path.basename(output_file)}")
    return count


print("✅ Conversion functions defined.")

In [ ]:
# ============================================================
# 2.2 Load or Convert Dataset
# ============================================================

# Check if clean JSONL files already exist
t5_files_exist = all(
    os.path.exists(f) for f in [T5_TRAIN, T5_VALID, T5_TEST]
)

if t5_files_exist:
    print("✅ Clean T5 JSONL files found! Skipping conversion.")
    for f in [T5_TRAIN, T5_VALID, T5_TEST]:
        line_count = sum(1 for _ in open(f, 'r', encoding='utf-8'))
        print(f"   {os.path.basename(f)}: {line_count:,} samples")
else:
    print("⏳ Clean files not found. Converting from raw JSON...")

    # Verify raw files exist
    for raw_file in [RAW_TRAIN, RAW_VALID, RAW_TEST]:
        if not os.path.exists(raw_file):
            raise FileNotFoundError(
                f"❌ Raw file not found: {raw_file}\n"
                f"   Please upload your dataset to: {DATASET_DIR}"
            )

    convert_raw_to_t5_jsonl(RAW_TRAIN, T5_TRAIN)
    convert_raw_to_t5_jsonl(RAW_VALID, T5_VALID)
    convert_raw_to_t5_jsonl(RAW_TEST, T5_TEST)

    print("\n✅ All conversions complete!")

In [ ]:
# ============================================================
# 2.3 Load Dataset with HuggingFace Datasets
# ============================================================

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": T5_TRAIN,
        "validation": T5_VALID,
        "test": T5_TEST
    }
)

print("\n📊 Dataset Summary:")
print("=" * 40)
for split_name, split_data in dataset.items():
    print(f"  {split_name:12s}: {len(split_data):>8,} samples")
print("=" * 40)

In [ ]:
# ============================================================
# 2.4 Optional: Sample Dataset for Quick Testing
# ============================================================

if SAMPLE_RATIO < 1.0:
    print(f"⚡ Sampling {SAMPLE_RATIO*100:.0f}% of the data for faster training...")

    for split_name in dataset:
        n_original = len(dataset[split_name])
        n_sampled = max(100, int(n_original * SAMPLE_RATIO))

        dataset[split_name] = dataset[split_name].shuffle(
            seed=SEED
        ).select(range(n_sampled))

        print(f"  {split_name}: {n_original:,} → {n_sampled:,}")

    print("\n✅ Sampling complete.")
else:
    print("✅ Using full dataset (SAMPLE_RATIO = 1.0)")

In [ ]:
# ============================================================
# 2.5 Inspect a Sample
# ============================================================

sample = dataset["train"][0]

print("📝 Input (first 500 chars):")
print("-" * 50)
print(sample["input"][:500])

print(f"\n🎯 Target:")
print("-" * 50)
print(sample["target"])

---
## 3. 🔤 Tokenization

In [ ]:
# ============================================================
# 3.1 Load Tokenizer & Add Special Tokens
# ============================================================

from transformers import AutoTokenizer

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Add hierarchical classification tokens
special_tokens = {
    "additional_special_tokens": [
        "<main_class>",
        "<sub_class>"
    ]
}
num_added = tokenizer.add_special_tokens(special_tokens)

print(f"\n✅ Tokenizer loaded.")
print(f"   Original vocab : {len(tokenizer) - num_added:,}")
print(f"   Added tokens   : {num_added}")
print(f"   New vocab size : {len(tokenizer):,}")

In [ ]:
# ============================================================
# 3.2 Verify Special Tokens
# ============================================================

test_text = "<main_class>تعليم <sub_class>جامعات سعودية"
tokens = tokenizer.tokenize(test_text)

print(f"Test: {test_text}")
print(f"Tokens: {tokens}")

# Verify round-trip
encoded = tokenizer(test_text, return_tensors="pt")
decoded = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=False)
print(f"Decoded: {decoded}")

In [ ]:
# ============================================================
# 3.3 Tokenize Dataset (or Load from Cache)
# ============================================================


def preprocess_function(examples):
    """Tokenize input text and target labels."""

    model_inputs = tokenizer(
        examples["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# Check for cached tokenized dataset
if os.path.exists(TOKENIZED_DIR) and SAMPLE_RATIO == 1.0:
    from datasets import load_from_disk

    print("✅ Loading cached tokenized dataset from Drive...")
    tokenized_dataset = load_from_disk(TOKENIZED_DIR)
    print(tokenized_dataset)
else:
    print("⏳ Tokenizing dataset (this may take a few minutes)...")

    tokenized_dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["input", "target"],
        desc="Tokenizing"
    )

    # Save to Drive for future sessions (only for full dataset)
    if SAMPLE_RATIO == 1.0:
        print("\n💾 Saving tokenized dataset to Drive for caching...")
        tokenized_dataset.save_to_disk(TOKENIZED_DIR)
        print("✅ Cached!")

print("\n📊 Tokenized Dataset:")
print(tokenized_dataset)

In [ ]:
# ============================================================
# 3.4 Verify Tokenization
# ============================================================

sample = tokenized_dataset["train"][0]

print("📥 Input tokens (decoded):")
print("-" * 50)
decoded_input = tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=False
)
print(decoded_input[:300] + "..." if len(decoded_input) > 300 else decoded_input)

print(f"\n🎯 Label tokens (decoded):")
print("-" * 50)
print(tokenizer.decode(sample["labels"], skip_special_tokens=False))

print(f"\n📏 Lengths:")
print(f"   input_ids : {len(sample['input_ids'])}")
print(f"   labels    : {len(sample['labels'])}")

---
## 4. 🧠 Model Setup

| Strategy | What's Trained | Params | Speed |
|----------|---------------|--------|-------|
| **LoRA** | Small adapter matrices injected into attention layers | ~2-5% | ⚡⚡⚡ |
| **Freeze Encoder** | Decoder + Embeddings (encoder frozen) | ~50% | ⚡⚡ |
| **Full** | All layers (encoder + decoder + embeddings) | 100% | ⚡ |

In [ ]:
# ============================================================
# 4.1 Load Model & Apply Training Strategy
# ============================================================

from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
import gc

# Free any existing GPU memory
gc.collect()
torch.cuda.empty_cache()

print("Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Resize embeddings for new special tokens (ALWAYS needed)
model.resize_token_embeddings(len(tokenizer))


# ===========================================
# Apply Training Strategy
# ===========================================

if TRAINING_STRATEGY == "lora":
    # -----------------------------------------
    # LoRA: Inject small trainable adapters
    # -----------------------------------------
    from peft import (
        get_peft_model,
        LoraConfig,
        TaskType,
        prepare_model_for_kbit_training
    )

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=["q", "v"],  # Attention Q and V matrices
        bias="none",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Ensure new embedding tokens are also trainable
    for param in model.base_model.model.shared.parameters():
        param.requires_grad = True

    strategy_label = "LoRA"


elif TRAINING_STRATEGY == "freeze_encoder":
    # -----------------------------------------
    # Freeze Encoder: Only train decoder + embeddings
    # -----------------------------------------

    # Freeze all encoder layers
    for param in model.encoder.parameters():
        param.requires_grad = False

    # Keep decoder trainable (already is by default)
    # Keep shared embeddings trainable (for new special tokens)
    for param in model.shared.parameters():
        param.requires_grad = True

    # Enable gradient checkpointing on decoder only
    model.gradient_checkpointing_enable()

    strategy_label = "Freeze Encoder"


else:  # "full"
    # -----------------------------------------
    # Full Fine-tuning: Train everything
    # -----------------------------------------
    model.gradient_checkpointing_enable()
    strategy_label = "Full Fine-tune"


# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
frozen_params = total_params - trainable_params
trainable_pct = trainable_params / total_params * 100

print(f"\n{'=' * 55}")
print(f"  Strategy        : {strategy_label}")
print(f"  Device          : {device.upper()}")
print(f"  Total params    : {total_params / 1e6:.1f}M")
print(f"  Trainable       : {trainable_params / 1e6:.1f}M ({trainable_pct:.1f}%)")
print(f"  Frozen          : {frozen_params / 1e6:.1f}M ({100 - trainable_pct:.1f}%)")

if TRAINING_STRATEGY == "lora":
    print(f"  LoRA rank       : {LORA_R}")
    print(f"  LoRA alpha      : {LORA_ALPHA}")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"  GPU Memory Used : {allocated:.2f} GB")

print(f"{'=' * 55}")

In [ ]:
# ============================================================
# 4.2 Data Collator
# ============================================================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Quick sanity check
batch = data_collator([
    tokenized_dataset["train"][0],
    tokenized_dataset["train"][1]
])

print("✅ DataCollator test:")
for k, v in batch.items():
    print(f"   {k:15s}: {v.shape}")

In [ ]:
# ============================================================
# 4.3 Forward Pass Sanity Check
# ============================================================

with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        labels=batch["labels"].to(device)
    )

print(f"✅ Forward pass successful!")
print(f"   Initial loss: {outputs.loss.item():.4f}")

---
## 5. 🏋️ Training

> **T4 Optimization Notes:**
> - `gradient_checkpointing` saves ~40% VRAM
> - `gradient_accumulation_steps=4` gives effective batch size of 16
> - Precision is auto-detected (fp16 for T4, bf16 for Ampere+)
> - Checkpoints saved to Drive every 500 steps for resume support

In [ ]:
# ============================================================
# 5.1 Metrics Functions
# ============================================================

import re


def extract_labels(text):
    """Extract main and sub labels from generated text."""

    main = ""
    sub = ""

    main_match = re.search(
        r"<main_class>\s*(.*?)\s*<sub_class>",
        text, re.DOTALL
    )
    sub_match = re.search(
        r"<sub_class>\s*(.*)",
        text, re.DOTALL
    )

    if main_match:
        main = main_match.group(1).strip()
    if sub_match:
        sub = sub_match.group(1).strip()

    return main, sub


def compute_metrics(eval_pred):
    """Compute accuracy metrics during training evaluation."""

    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=False
    )

    # Replace -100 with pad token for decoding
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=False
    )

    total = len(decoded_labels)
    exact, main_correct, sub_correct = 0, 0, 0

    for pred, label in zip(decoded_preds, decoded_labels):
        pred_main, pred_sub = extract_labels(pred)
        true_main, true_sub = extract_labels(label)

        if pred_main == true_main:
            main_correct += 1
        if pred_sub == true_sub:
            sub_correct += 1
        if pred_main == true_main and pred_sub == true_sub:
            exact += 1

    return {
        "main_accuracy": round(main_correct / total, 4),
        "sub_accuracy": round(sub_correct / total, 4),
        "exact_match": round(exact / total, 4)
    }


print("✅ Metrics functions defined.")

In [ ]:
# ============================================================
# 5.2 Training Arguments (Strategy-Aware)
# ============================================================

from transformers import Seq2SeqTrainingArguments

# Auto-detect precision
use_fp16 = False
use_bf16 = False

if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    if capability[0] >= 8:
        use_bf16 = True
        precision_label = "bf16 (Ampere+)"
    elif capability[0] >= 7:
        use_fp16 = False  # Fixed: T5 does not support fp16 well (leads to NaN loss).\n        precision_label = "fp32 (forced, no fp16 for T5)"\n    else:
        precision_label = "fp32 (no mixed precision)"
else:
    precision_label = "fp32 (CPU)"

print(f"Auto-detected precision: {precision_label}")

# Adjust save/eval frequency based on strategy
if TRAINING_STRATEGY == "lora":
    eval_save_steps = 500    # LoRA trains faster
    logging_freq = 50
elif TRAINING_STRATEGY == "freeze_encoder":
    eval_save_steps = 500
    logging_freq = 50
else:  # full
    eval_save_steps = 500
    logging_freq = 100

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # ========================
    # Training
    # ========================
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,

    # ========================
    # Batch Size
    # ========================
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # ========================
    # GPU Precision (auto-detected)
    # ========================
    fp16=use_fp16,
    bf16=use_bf16,

    # ========================
    # Evaluation & Checkpoints
    # ========================
    eval_strategy="steps",
    eval_steps=eval_save_steps,
    save_strategy="steps",
    save_steps=eval_save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ========================
    # Generation during Eval
    # ========================
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,  # Greedy (fast on T4)

    # ========================
    # Memory Optimization
    # ========================
    eval_accumulation_steps=10,

    # ========================
    # Logging
    # ========================
    logging_strategy="steps",
    logging_steps=logging_freq,
    report_to="none",

    seed=SEED
)

print(f"\nTraining arguments configured.")
print(f"   Strategy        : {TRAINING_STRATEGY}")
print(f"   Effective batch : {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   Epochs          : {NUM_EPOCHS}")
print(f"   Learning rate   : {LEARNING_RATE}")
print(f"   Eval/Save every : {eval_save_steps} steps")
print(f"   Precision       : {precision_label}")

In [ ]:
# ============================================================
# 5.3 Create Trainer
# ============================================================

from transformers import Seq2SeqTrainer, EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.001
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

print("✅ Trainer created with EarlyStopping (patience=3).")

In [ ]:
# ============================================================
# 5.4 Train! (with automatic checkpoint resume)
# ============================================================

# Find latest checkpoint if resume is enabled
resume_checkpoint = None

if RESUME_FROM_CHECKPOINT and os.path.exists(OUTPUT_DIR):
    checkpoints = sorted(
        [
            d for d in os.listdir(OUTPUT_DIR)
            if d.startswith("checkpoint-")
            and os.path.isdir(os.path.join(OUTPUT_DIR, d))
        ],
        key=lambda x: int(x.split("-")[1])
    )

    if checkpoints:
        resume_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])
        print(f"🔄 Resuming from checkpoint: {checkpoints[-1]}")
    else:
        print("🆕 No checkpoint found. Starting fresh training.")
else:
    print("🆕 Starting fresh training.")

print("\n" + "=" * 55)
print("  🏋️ TRAINING STARTED")
print("=" * 55)

train_result = trainer.train(
    resume_from_checkpoint=resume_checkpoint
)

print("\n" + "=" * 55)
print("  ✅ TRAINING COMPLETE")
print("=" * 55)
print(f"  Total steps    : {train_result.global_step}")
print(f"  Training loss  : {train_result.training_loss:.4f}")

if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Peak GPU memory: {peak_mem:.2f} GB")

---
## 6. 📊 Evaluation

In [ ]:
# ============================================================
# 6.1 Evaluate on Validation Set
# ============================================================

print("⏳ Evaluating on validation set...")
eval_results = trainer.evaluate()

print("\n📊 Validation Results:")
print("=" * 45)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:.4f}")
    else:
        print(f"  {key:25s}: {value}")
print("=" * 45)

In [ ]:
# ============================================================
# 6.2 Full Test Set Evaluation with Classification Report
# ============================================================

from sklearn.metrics import classification_report
from collections import Counter

print("⏳ Running predictions on test set...")

test_predictions = trainer.predict(tokenized_dataset["test"])

# Decode predictions and labels
preds = test_predictions.predictions
if isinstance(preds, tuple):
    preds = preds[0]

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=False)

test_labels = np.where(
    test_predictions.label_ids != -100,
    test_predictions.label_ids,
    tokenizer.pad_token_id
)
decoded_labels = tokenizer.batch_decode(test_labels, skip_special_tokens=False)

# Extract structured labels
true_mains, pred_mains = [], []
true_subs, pred_subs = [], []

for pred, label in zip(decoded_preds, decoded_labels):
    pm, ps = extract_labels(pred)
    tm, ts = extract_labels(label)
    pred_mains.append(pm)
    true_mains.append(tm)
    pred_subs.append(ps)
    true_subs.append(ts)

# Main category report
print("\n" + "=" * 60)
print("  📊 MAIN CATEGORY CLASSIFICATION REPORT")
print("=" * 60)
print(
    classification_report(
        true_mains, pred_mains,
        zero_division=0
    )
)

# Subcategory report (top categories)
print("\n" + "=" * 60)
print("  📊 SUBCATEGORY CLASSIFICATION REPORT (top 30)")
print("=" * 60)

# Get unique subcategories, sorted by frequency
sub_counts = Counter(true_subs)
top_subs = [s for s, _ in sub_counts.most_common(30)]

# Filter to top subcategories for readable report
filtered_true = []
filtered_pred = []
for t, p in zip(true_subs, pred_subs):
    if t in top_subs:
        filtered_true.append(t)
        filtered_pred.append(p)

print(
    classification_report(
        filtered_true, filtered_pred,
        zero_division=0
    )
)

# Overall metrics
exact = sum(
    1 for pm, ps, tm, ts in zip(pred_mains, pred_subs, true_mains, true_subs)
    if pm == tm and ps == ts
)
total = len(true_mains)

print("\n" + "=" * 45)
print(f"  Main Accuracy  : {sum(1 for p,t in zip(pred_mains, true_mains) if p==t) / total:.4f}")
print(f"  Sub Accuracy   : {sum(1 for p,t in zip(pred_subs, true_subs) if p==t) / total:.4f}")
print(f"  Exact Match    : {exact / total:.4f}")
print(f"  Total Samples  : {total:,}")
print("=" * 45)

In [ ]:
# ============================================================
# 6.3 Confusion Matrix (Main Categories)
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get unique main categories
unique_mains = sorted(set(true_mains))

if len(unique_mains) <= 20:  # Only plot if manageable size
    cm = confusion_matrix(
        true_mains, pred_mains,
        labels=unique_mains
    )

    fig, ax = plt.subplots(figsize=(12, 10))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=unique_mains
    )
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
    ax.set_title("Main Category Confusion Matrix", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️  Too many categories ({len(unique_mains)}) for confusion matrix.")
    print("   Showing top 10 error pairs instead:")

    errors = [
        (t, p) for t, p in zip(true_mains, pred_mains) if t != p
    ]
    error_counts = Counter(errors)
    print("\n   True → Predicted : Count")
    for (true, pred), count in error_counts.most_common(10):
        print(f"   {true} → {pred}: {count}")

In [ ]:
# ============================================================
# 6.4 Error Analysis (Sample Misclassifications)
# ============================================================

print("🔍 Sample Misclassifications:")
print("=" * 70)

error_count = 0
for i, (pm, ps, tm, ts) in enumerate(
    zip(pred_mains, pred_subs, true_mains, true_subs)
):
    if pm != tm or ps != ts:
        error_count += 1
        if error_count <= 10:  # Show first 10 errors
            print(f"\n  Sample #{i}:")
            print(f"    True : {tm} > {ts}")
            print(f"    Pred : {pm} > {ps}")

print(f"\n  Total errors: {error_count}/{total} ({error_count/total*100:.1f}%)")

---
## 7. 🔮 Inference

Use this section to classify new articles after training.

In [ ]:
# ============================================================
# 7.1 Inference Function
# ============================================================

# Get the actual model for inference (handles LoRA wrapper)
inference_model = model


def classify_article(
    title,
    content,
    _model=inference_model,
    _tokenizer=tokenizer,
    max_input_length=MAX_INPUT_LENGTH,
    num_beams=4
):
    """
    Classify an Arabic article into main and sub categories.

    Args:
        title: Article title
        content: Article content
        num_beams: Beam search width (higher = better but slower)

    Returns:
        dict with main_category, sub_category, and raw_output
    """

    # Build input prompt
    input_text = (
        "المهمة: تصنيف هرمي\n\n"
        f"العنوان:\n{title}\n\n"
        f"النص:\n{content}"
    )

    # Tokenize
    inputs = _tokenizer(
        input_text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(_model.device)

    # Generate
    with torch.no_grad():
        outputs = _model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=num_beams,
            early_stopping=True
        )

    # Decode
    raw_output = _tokenizer.decode(
        outputs[0], skip_special_tokens=False
    )

    # Extract labels
    main_cat, sub_cat = extract_labels(raw_output)

    return {
        "main_category": main_cat,
        "sub_category": sub_cat,
        "raw_output": raw_output
    }


print("Inference function ready.")

In [ ]:
# ============================================================
# 7.2 Test Inference with a Sample
# ============================================================

# Use first test sample
test_sample = dataset["test"][0]

# Parse title and content from the input text
input_text = test_sample["input"]
lines = input_text.split("\n")

# Find title and content
title_idx = next(
    (i for i, l in enumerate(lines) if "العنوان" in l),
    None
)
content_idx = next(
    (i for i, l in enumerate(lines) if "النص" in l),
    None
)

if title_idx is not None and content_idx is not None:
    sample_title = "\n".join(
        lines[title_idx + 1 : content_idx]
    ).strip()
    sample_content = "\n".join(
        lines[content_idx + 1 :]
    ).strip()
else:
    # Fallback
    sample_title = lines[0] if lines else ""
    sample_content = "\n".join(lines[1:]) if len(lines) > 1 else ""

# Run inference
result = classify_article(sample_title, sample_content)

print("📝 Title:")
print(f"   {sample_title[:100]}..." if len(sample_title) > 100 else f"   {sample_title}")

print(f"\n🎯 True label:")
print(f"   {test_sample['target']}")

print(f"\n🤖 Predicted:")
print(f"   Main: {result['main_category']}")
print(f"   Sub:  {result['sub_category']}")
print(f"   Raw:  {result['raw_output']}")

In [ ]:
# ============================================================
# 7.3 Batch Inference Helper
# ============================================================


def classify_batch(articles, batch_size=8):
    """
    Classify a batch of articles efficiently.

    Args:
        articles: List of dicts with "title" and "content" keys
        batch_size: Processing batch size

    Returns:
        List of classification results
    """

    results = []

    for i in range(0, len(articles), batch_size):
        batch = articles[i:i + batch_size]

        input_texts = [
            f"المهمة: تصنيف هرمي\n\nالعنوان:\n{a["title"]}\n\nالنص:\n{a["content"]}"
            for a in batch
        ]

        inputs = tokenizer(
            input_texts,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=1
            )

        decoded = tokenizer.batch_decode(
            outputs, skip_special_tokens=False
        )

        for text in decoded:
            main, sub = extract_labels(text)
            results.append({
                "main_category": main,
                "sub_category": sub
            })

    return results


print("✅ Batch inference function ready.")
print("\nUsage:")
print('  results = classify_batch([{"title": "...", "content": "..."}])')

---
## 8. 💾 Export & Save

In [ ]:
# ============================================================
# 8.1 Save Final Model to Google Drive
# ============================================================

import shutil

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print("Saving model to Google Drive...")

if TRAINING_STRATEGY == "lora":
    # Save LoRA adapter weights (small ~20MB)
    model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)

    # Also save merged model for easy inference
    merged_dir = os.path.join(OUTPUT_DIR, "merged_model")
    os.makedirs(merged_dir, exist_ok=True)

    print("\nMerging LoRA weights into base model...")
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print(f"  Merged model saved to: {merged_dir}")
else:
    trainer.save_model(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Verify
saved_files = os.listdir(FINAL_MODEL_DIR)
total_size = sum(
    os.path.getsize(os.path.join(FINAL_MODEL_DIR, f))
    for f in saved_files
    if os.path.isfile(os.path.join(FINAL_MODEL_DIR, f))
) / 1e6

print(f"\nModel saved to: {FINAL_MODEL_DIR}")
print(f"   Files: {len(saved_files)}")
print(f"   Total size: {total_size:.1f} MB")
print(f"\n   Saved files:")
for f in sorted(saved_files):
    fpath = os.path.join(FINAL_MODEL_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"   - {f} ({size:.1f} MB)")

In [ ]:
# ============================================================
# 8.2 (Optional) Upload to HuggingFace Hub
# ============================================================

UPLOAD_TO_HF = False  # Set to True to upload
HF_REPO_NAME = "your-username/arat5-almersal-classifier"  # Change this!

if UPLOAD_TO_HF:
    from huggingface_hub import notebook_login

    # Login (will prompt for token)
    notebook_login()

    print(f"\nUploading to: {HF_REPO_NAME}...")

    if TRAINING_STRATEGY == "lora":
        # Upload merged model for easy use
        from transformers import AutoModelForSeq2SeqLM as LoadModel
        merged_dir = os.path.join(OUTPUT_DIR, "merged_model")
        upload_model = LoadModel.from_pretrained(merged_dir)
        upload_model.push_to_hub(
            HF_REPO_NAME,
            commit_message="Upload AraT5 classifier (LoRA merged)"
        )
    else:
        model.push_to_hub(
            HF_REPO_NAME,
            commit_message="Upload AraT5 hierarchical classifier"
        )

    tokenizer.push_to_hub(
        HF_REPO_NAME,
        commit_message="Upload tokenizer with special tokens"
    )

    print(f"\nModel uploaded to: https://huggingface.co/{HF_REPO_NAME}")
else:
    print("HuggingFace upload skipped.")
    print("   Set UPLOAD_TO_HF = True and update HF_REPO_NAME to upload.")

In [ ]:
# ============================================================
# 8.3 Load Saved Model (for future use)
# ============================================================

# Uncomment the appropriate section based on your training strategy:

# --- Option A: Load merged model (works for all strategies) ---
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#
# MODEL_PATH = "/content/drive/MyDrive/arat5_classifier/merged_model"  # or final_model
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
# model.to("cuda" if torch.cuda.is_available() else "cpu")
# print("Model loaded!")

# --- Option B: Load LoRA adapter (smaller, needs base model) ---
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from peft import PeftModel
#
# MODEL_NAME = "UBC-NLP/AraT5-base"
# ADAPTER_PATH = "/content/drive/MyDrive/arat5_classifier/final_model"
#
# tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
# base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
# base_model.resize_token_embeddings(len(tokenizer))
# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# model = model.merge_and_unload()  # Optional: merge for faster inference
# model.to("cuda" if torch.cuda.is_available() else "cpu")
# print("LoRA model loaded!")

print("Code above is ready to use in a new session.")

---
## 📋 Training Summary

| Item | Details |
|------|--------|
| **Model** | AraT5-base (UBC-NLP) |
| **Task** | Hierarchical Classification |
| **Special Tokens** | `<main_class>`, `<sub_class>` |
| **Max Input Length** | 512 tokens |
| **Max Target Length** | 16 tokens |
| **Optimizer** | AdamW (cosine scheduler) |
| **Mixed Precision** | Auto-detected |
| **Early Stopping** | Patience = 3 |

### 🎯 Training Strategies

| Strategy | Params Trained | Time (T4) | VRAM | Best For |
|----------|---------------|-----------|------|----------|
| **LoRA** | ~2-5% (~4M) | 3-5h | ~6GB | Colab Free ✅ |
| **Freeze Encoder** | ~50% (~110M) | 5-7h | ~10GB | Balanced |
| **Full** | 100% (~220M) | 24-36h | ~14GB | Max accuracy |

### ⚡ T4 Colab Tips
- **Recommended:** Use `TRAINING_STRATEGY = "lora"` for Colab Free
- If you run out of time, training will auto-resume from the last checkpoint
- Use `SAMPLE_RATIO = 0.1` for quick testing before full training
- If you get OOM errors, reduce `TRAIN_BATCH_SIZE` to 2
- LoRA adapter is only ~20MB vs ~900MB for full model